# Speech and Audio Processing with Deep Learning
Deep learning has revolutionized speech recognition, synthesis, music generation, and environmental sound classification. This notebook covers audio representations, WaveNet, CTC loss, Conformer, and Whisper.

## 1. Audio Representations
Raw audio is a 1D waveform sampled at 16kHz or 22kHz. Deep learning models work with several derived representations:

| Representation | Description | Used For |
|---|---|---|
| Raw Waveform | 1D amplitude samples | WaveNet, end-to-end models |
| Spectrogram | 2D time-frequency energy map | CNN-based models |
| Mel Spectrogram | Frequency bins on mel scale (perceptual) | Most speech/audio DL models |
| MFCC | Mel frequency cepstral coefficients | Traditional ASR, speaker ID |
| Log-Mel Spectrogram | Log of mel spectrogram | Whisper, Conformer |

## 2. WaveNet (DeepMind, 2016)
A deep autoregressive model for raw audio waveform generation.
- Models P(x_t | x_1, ..., x_{t-1}) — the probability of each audio sample given all previous samples
- Uses dilated causal convolutions to achieve very large receptive fields efficiently
- Dilation factors: 1, 2, 4, 8, 16, 32 — exponentially increasing context with few parameters
- Gated activation: tanh(W_f * x) * sigmoid(W_g * x)
- Used for text-to-speech and music generation; sounds extremely natural

In [1]:
import tensorflow as tf
from tensorflow.keras import layers

def dilated_causal_conv_block(x, filters, dilation_rate, residual_channels=None):
    # Causal convolution: padding only on the left (past)
    conv = layers.Conv1D(filters * 2, kernel_size=2,
                         dilation_rate=dilation_rate,
                         padding='causal')(x)
    # Gated activation (WaveNet style)
    tanh_out = tf.math.tanh(conv[:, :, :filters])
    sigmoid_out = tf.math.sigmoid(conv[:, :, filters:])
    gated = tanh_out * sigmoid_out
    # Residual connection
    if residual_channels:
        residual = layers.Conv1D(residual_channels, 1)(gated)
        return gated + x if x.shape[-1] == residual_channels else gated
    return gated

print("WaveNet dilated causal conv block defined.")
print("Dilation schedule: [1, 2, 4, 8, 16, 32] per stack")
print("Receptive field grows exponentially without parameter explosion.")

WaveNet dilated causal conv block defined.
Dilation schedule: [1, 2, 4, 8, 16, 32] per stack
Receptive field grows exponentially without parameter explosion.


## 3. Connectionist Temporal Classification (CTC)
The key loss function enabling end-to-end speech recognition without frame-level labels.

**Problem**: The input (audio frames) and output (text characters) don't align — we don't know which frames correspond to which characters.

**Solution**: CTC marginalizes over all possible alignments, allowing training with only (audio, transcript) pairs.
- A special blank token handles silence and repeated symbols
- CTC decoding: greedy or beam search, collapsing consecutive identical tokens and removing blanks
- Used in: DeepSpeech, Wav2Vec 2.0, and many industrial ASR systems

## 4. Conformer (2020)
Combines CNN (local feature extraction) with Transformer (global attention) for state-of-the-art ASR.
- **Architecture**: Multi-Head Self-Attention + Convolution Module + Feed-Forward Module + Layer Norm
- CNNs capture local acoustic patterns (phonemes); Transformers capture long-range dependencies (prosody)
- Dominates speech recognition benchmarks (LibriSpeech, CommonVoice)

## 5. Whisper (OpenAI, 2022)
An end-to-end speech recognition Transformer trained on 680,000 hours of multilingual audio.
- **Architecture**: CNN feature encoder (log-mel spectrogram → feature map) + Transformer Encoder-Decoder
- **Input**: 30-second log-mel spectrogram windows
- **Output**: Text tokens decoded autoregressively
- Supports 99 languages; handles noisy, accented, and spontaneous speech robustly
- Zero-shot capable — no per-domain fine-tuning needed

In [2]:
# Whisper usage (requires openai-whisper package)
# pip install openai-whisper

import numpy as np

# Load audio as float32 waveform (sample rate 16000 Hz)
def create_log_mel_spectrogram(audio, sample_rate=16000, n_mels=80, n_fft=400, hop_length=160):
    # Conceptual: librosa.feature.melspectrogram + log transform
    print(f"Audio duration: {len(audio)/sample_rate:.2f}s")
    print(f"Output shape: ({n_mels}, {len(audio)//hop_length})")
    print("Whisper uses 30s windows -> (80, 3000) mel spectrograms")

# Simulate 3 seconds of audio at 16kHz
dummy_audio = np.random.randn(48000).astype(np.float32)
create_log_mel_spectrogram(dummy_audio)

# import whisper
# model = whisper.load_model("base")
# result = model.transcribe("audio.mp3")
# print(result["text"])

Audio duration: 3.00s
Output shape: (80, 300)
Whisper uses 30s windows -> (80, 3000) mel spectrograms


# Conclusions and Key Takeaways
- Audio is fundamentally a 1D temporal signal; log-mel spectrograms provide a perceptually meaningful 2D representation.
- WaveNet proved that raw waveform generation is possible; it enabled voice synthesis to reach near-human quality.
- CTC loss removed the need for expensive frame-level annotations, enabling end-to-end ASR training.
- Conformer's CNN+Transformer combination is the current state-of-the-art for speech recognition.
- Whisper demonstrated that massive scale + simple architecture achieves robust zero-shot multilingual ASR.

# Pros and Cons
**Pros:**
- Mel spectrograms are highly effective 2D inputs for CNN-based audio processing
- Transformer-based models (Whisper, Wav2Vec2) achieve robust zero-shot generalization to new domains
- CTC allows training with only weak supervision (audio + transcript pairs, no frame alignment)

**Cons:**
- Audio data is high-dimensional: 16000 samples per second; long files stress memory and compute
- WaveNet-style inference is inherently slow (one sample at a time); requires distillation for real-time TTS
- Speech models are sensitive to domain mismatch: background noise, accents, sampling rate

# 15 Interview Questions and Answers

1. **What is a Mel Spectrogram?**
   *Answer*: A spectrogram with frequency bins spaced on the Mel scale (logarithmic, matching human auditory perception). It maps short-time Fourier transform magnitudes to perceptually meaningful frequency bands.

2. **What is CTC loss and why is it important for speech recognition?**
   *Answer*: Connectionist Temporal Classification is a loss that sums the probabilities of all possible alignments between input frames and output characters, using a blank token to handle silence and repetition. It removes the need for frame-level label alignment.

3. **What is the dilated causal convolution in WaveNet?**
   *Answer*: A 1D convolution that skips over inputs with gaps (dilation), exponentially increasing the effective receptive field while maintaining causal (past-only) attention and keeping parameter count low.

4. **What is the Conformer architecture?**
   *Answer*: A hybrid encoder combining Multi-Head Self-Attention (Transformer block) with a convolution module (for local feature learning). The combination captures both local acoustic patterns and long-range dependencies simultaneously.

5. **What makes Whisper different from other ASR models?**
   *Answer*: Scale, data diversity, and simplicity. Trained on 680K hours of weakly supervised web-scraped audio in 99 languages with no domain-specific fine-tuning. Uses a straightforward CNN-Transformer encoder-decoder.

6. **What is Voice Activity Detection (VAD)?**
   *Answer*: A binary classification task that identifies whether each audio frame contains speech or non-speech (silence, background noise). Used as a preprocessing step in ASR pipelines.

7. **What is the difference between automatic speech recognition (ASR) and text-to-speech (TTS)?**
   *Answer*: ASR converts audio waveforms to text transcriptions. TTS (speech synthesis) converts text to audio waveforms. WaveNet is a TTS model; Whisper is ASR.

8. **What are MFCCs and why were they dominant before deep learning?**
   *Answer*: Mel Frequency Cepstral Coefficients — a compact feature vector computed from the log-mel spectrogram via Discrete Cosine Transform. They provide a compact, noise-robust representation that classical HMM-based ASR systems used effectively.

9. **What is Wav2Vec 2.0?**
   *Answer*: A self-supervised model pre-trained on raw waveforms using contrastive learning on quantized latent representations. Fine-tuned with CTC on small labeled datasets, it achieves state-of-the-art ASR with very few labeled examples.

10. **How does Text-to-Speech with Tacotron work?**
    *Answer*: Tacotron 2 uses a sequence-to-sequence model with attention to convert text to Mel spectrograms, then a WaveNet/WaveGlow vocoder converts the spectrogram to a raw audio waveform.

11. **What is the role of the vocoder in TTS?**
    *Answer*: A vocoder converts an intermediate representation (Mel spectrogram) into a raw audio waveform. Neural vocoders like WaveNet, WaveGlow, and HiFi-GAN generate high-fidelity audio.

12. **What is Speaker Diarization?**
    *Answer*: The task of segmenting an audio recording to identify "who spoke when". It does not transcribe speech but labels segments with speaker identities, often using speaker embedding models.

13. **What is Transfer Learning for audio?**
    *Answer*: Pre-training on large audio datasets (like AudioSet for sound classification or LibriSpeech for ASR) then fine-tuning on a smaller target domain. Wav2Vec and Whisper are both examples of powerful pre-trained audio models.

14. **How are noisy environments handled in speech models?**
    *Answer*: Through noise-robust training (multi-condition training with added noise), data augmentation (SpecAugment: time/frequency masking), or separate enhancement models (denoising using spectral subtraction or RNN-based speech enhancement).

15. **What is SpecAugment?**
    *Answer*: A simple but highly effective data augmentation technique for speech: randomly masking consecutive time steps (time masking) and consecutive frequency bins (frequency masking) on the spectrogram before training. Greatly reduces overfitting.
